# О-информация: совместное обучение, погрешности и альтернативная оценка

Небольшой самостоятельный эксперимент по мотивам **SΩI (Bounoua et al., 2024)**.
Это **не воспроизведение результатов официального репозитория**. Здесь компактные MLP,
шесть скалярных гауссовских переменных, аддитивный гауссовский шум и одинаковый
Gaussian skip connection для всех моделей. Цель — проверить механизм и подготовить
осмысленную гипотезу для дальнейшего эксперимента с оригинальным кодом.

**Что сравниваем:**

1. `shared`: одна сеть для всех задач восстановления.
2. `split_budget`: три сети — joint, marginal и conditional; суммарное число параметров
   немного меньше, чем у `shared` (6156 против 6342 при ширине 64).
3. `split_full`: три сети той же ширины, что у `shared` (19026 параметров).

Разделение на три семейства — промежуточный вариант. Это ещё не $2N+1$ независимых сетей:
внутри marginal и conditional задачами разных переменных по-прежнему делятся параметры.
Для каждой модели вычисляются **две оценки по одним и тем же тестовым данным и шумам**:
разность квадратов предсказаний (`direct`) и разность ошибок восстановления (`risk`).

**Запуск:** выполните ячейки сверху вниз, затем нажмите «Запустить / продолжить» на панели.
В Colab GPU можно выбрать через «Среда выполнения → Сменить среду выполнения».
На Mac выбирается MPS, если он доступен; CPU тоже поддерживается. Данные генерируются локально.
По умолчанию в Colab предлагается сохранить результаты на Google Drive: согласие выдаётся
в интерфейсе Colab при подключении диска. Если отключить Drive, скачивайте ZIP после запуска.

Все основные выводы пока являются **гипотезами**. Один короткий запуск показывает работоспособность,
но не устанавливает преимущество метода. В ноутбуке намеренно нет заранее записанного «победителя».


In [ ]:
# Зависимости. В Colab большая часть обычно уже установлена.
import importlib.util
import subprocess
import sys
packages = ["numpy", "scipy", "pandas", "matplotlib", "torch", "ipywidgets", "tqdm"]
missing = [name for name in packages if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
print("Colab:", IN_COLAB)


In [ ]:
from pathlib import Path
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/soi_experiment")
elif IN_COLAB:
    OUTPUT_ROOT = Path("/content/soi_experiment")
else:
    OUTPUT_ROOT = Path.cwd() / "soi_experiment"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Результаты:", OUTPUT_ROOT)
print("При отключённом Drive файлы Colab могут исчезнуть после завершения сеанса.")


## Математика эксперимента

Пусть $Y=X+\sigma Z$, $Z\sim\mathcal N(0,I)$, $u=\log\sigma$.
Обозначим предсказания шума

$$J_i=\mathbb E[Z_i\mid Y],\qquad M_i=\mathbb E[Z_i\mid Y_i],\qquad
C_i=\mathbb E[Z_i\mid Y_i,X_{-i}].$$

Замена $t=\sigma^2/2$ в формулах SΩI даёт

$$TC=\int_{-\infty}^{\infty}\mathbb E\|J-M\|^2du,\quad
DTC=\int_{-\infty}^{\infty}\mathbb E\|J-C\|^2du,\quad \Omega=TC-DTC.$$

Определим $R_A=\mathbb E\|Z-A\|^2$. Для точных УМО

$$\Omega=\int(R_M+R_C-2R_J)\,du.$$

Именно эту вторую формулу проверяет `risk`. В силу ортогональности ошибки УМО
$\mathbb E\|Z-\widehat A\|^2=R_A+\mathbb E\|\widehat A-A\|^2$.
Поэтому для фиксированных обученных сетей, независимого теста и того же конечного
интервала интегрирования:

$$\mathbb E_{test}[\widehat\Omega_{risk}]-\Omega_{interval}
=E_M+E_C-2E_J,\quad
E_A=\int\mathbb E\|\widehat A-A\|^2du.$$

Это точное разложение смещения из-за приближённых УМО, а не доказательство общей состоятельности.
К полной ошибке добавляются отсечение диапазона шума, квадратура и случайность теста.
Меньшая чувствительность смещения к ошибке сети **не гарантирует меньшую дисперсию**.
Эта конструкция родственна I-MMSE и MMG; её нельзя заявлять как новый общий принцип.

Для многомерного нормального распределения с ковариацией $\Sigma$ доступны точные УМО
и истинные значения. При скалярных переменных:

$$TC=\tfrac12\left(\sum_i\log\Sigma_{ii}-\log\det\Sigma\right),\quad
DTC=\tfrac12\left(\log\det\Sigma+\sum_i\log(\Sigma^{-1})_{ii}\right).$$

Сеть не получает $\Sigma$. Она используется только при генерации и внешней проверке.
В одинаковом для всех моделей skip connection использовано знание единичных маргинальных
дисперсий этого синтетического семейства. Поэтому перенос результатов на произвольные данные
требует отдельной проверки.


## Реализация

Код ниже можно читать по функциям: `covariance` — распределения, `Denoiser` — сети,
`evaluate` — оценки и диагностика, `run_one` — обучение и checkpoint, `run_suite` — серии запусков.
Во всех архитектурах одинаковы данные, последовательность минибатчей, уровни шума и целевые шумы.
Каждый шаг обучает все три семейства с одинаковым весом. Обучение имеет фиксированное число шагов;
истинная О-информация и финальный тест не используются для выбора checkpoint.
Время и число параметров записываются: одинаковые шаги не означают одинаковое время.

Интегрирование: трапеции по $u=\log\sigma$ и Monte Carlo по тестовым наблюдениям/шумам.
Один тестовый объект участвует во всех уровнях шума; стандартная ошибка считается **после**
интегрирования вклада каждого объекта. Поэтому повторное использование объектов не выдаётся
за увеличение размера независимой выборки.


In [ ]:
import copy
import hashlib
import json
import math
import os
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
from scipy.optimize import brentq
import torch
from torch import nn

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterator, **kwargs):
        return iterator

VERSION = "soi-ablation-1.0"


def gaussian_truth(cov):
    sign, logdet = np.linalg.slogdet(cov)
    if sign <= 0 or np.linalg.eigvalsh(cov).min() <= 0:
        raise ValueError("Covariance must be positive definite")
    precision = np.linalg.inv(cov)
    tc = 0.5 * (np.log(np.diag(cov)).sum() - logdet)
    dtc = 0.5 * (logdet + np.log(np.diag(precision)).sum())
    return dict(tc=float(tc), dtc=float(dtc), omega=float(tc - dtc))


def covariance(case):
    def red(r):
        return (1 - r) * np.eye(3) + r * np.ones((3, 3))
    a = 0.95 / np.sqrt(2)
    syn = np.array([[1., 0., a], [0., 1., a], [a, a, 1.]])
    if case == "pair_null":
        return block_diag(*[np.array([[1., .9], [.9, 1.]])] * 3)
    if case == "independent":
        return np.eye(6)
    if case == "redundant":
        return block_diag(red(.85), red(.85))
    if case == "synergistic":
        return block_diag(syn, syn)
    if case in ("balanced", "mixed_positive", "mixed_negative"):
        r0 = brentq(lambda r: gaussian_truth(red(r))["omega"]
                    + gaussian_truth(syn)["omega"], 0., .999)
        delta = {"balanced": 0., "mixed_positive": .025,
                 "mixed_negative": -.025}[case]
        return block_diag(red(r0 + delta), syn)
    raise ValueError(case)


def trapezoid_weights(grid):
    w = np.empty_like(grid)
    w[0] = (grid[1] - grid[0]) / 2
    w[-1] = (grid[-1] - grid[-2]) / 2
    w[1:-1] = (grid[2:] - grid[:-2]) / 2
    return w


def oracle_integral(cov, sigma_min=.02, sigma_max=30., grid_size=512):
    """Exact expected Gaussian integrands; only quadrature/truncation remain."""
    u = np.linspace(np.log(sigma_min), np.log(sigma_max), grid_size)
    eigenvalues = np.linalg.eigvalsh(cov)
    marginal_var = np.diag(cov)
    conditional_var = 1 / np.diag(np.linalg.inv(cov))
    s2 = np.exp(2 * u)[:, None]
    rj = (eigenvalues / (eigenvalues + s2)).sum(axis=1)
    rm = (marginal_var / (marginal_var + s2)).sum(axis=1)
    rc = (conditional_var / (conditional_var + s2)).sum(axis=1)
    w = trapezoid_weights(u)
    return dict(tc=float(w @ (rm - rj)), dtc=float(w @ (rj - rc)),
                omega=float(w @ (rm + rc - 2 * rj)))


def parameter_count(width, n=6, groups=1):
    din = 3 * n + 9
    return groups * (width * width + width * (din + n + 2) + n)


def matched_width(shared_width, n=6):
    budget = parameter_count(shared_width, n)
    return max(w for w in range(1, shared_width + 1)
               if parameter_count(w, n, 3) <= budget)


In [ ]:
class Denoiser(nn.Module):
    def __init__(self, n, width, split, sigma_min, sigma_max):
        super().__init__()
        self.n, self.split = n, split
        self.umin, self.umax = math.log(sigma_min), math.log(sigma_max)
        self.networks = nn.ModuleList([
            nn.Sequential(nn.Linear(3 * n + 9, width), nn.SiLU(),
                          nn.Linear(width, width), nn.SiLU(), nn.Linear(width, n))
            for _ in range(3 if split else 1)
        ])
        for net in self.networks:
            nn.init.zeros_(net[-1].weight)
            nn.init.zeros_(net[-1].bias)

    def forward(self, x, noise, sigma):
        # The three families receive exactly the permitted observations.
        b, n = x.shape
        sigma = sigma.reshape(b, 1)
        y = x + sigma * noise
        yn = y / torch.sqrt(1 + sigma.square())
        eye = torch.eye(n, dtype=x.dtype, device=x.device)[:, None, :]
        joint = yn[None]
        marginal = yn[None] * eye
        conditional = x[None] * (1 - eye) + yn[None] * eye
        values = torch.cat([joint, marginal, conditional], dim=0)
        noisy = torch.cat([torch.ones_like(joint), eye.expand(-1, b, -1),
                           eye.expand(-1, b, -1)], dim=0)
        observed = torch.cat([torch.ones_like(joint), eye.expand(-1, b, -1),
                              torch.ones_like(conditional)], dim=0)
        u = 2 * (torch.log(sigma) - self.umin) / (self.umax - self.umin) - 1
        frequencies = torch.tensor([1., 2., 4., 8.], device=x.device)
        phase = math.pi * u * frequencies
        time_features = torch.cat([u, phase.sin(), phase.cos()], dim=-1)
        f = torch.cat([values, noisy, observed,
                       time_features[None].expand(2*n + 1, -1, -1)], dim=-1)
        if self.split:
            groups = (f[:1], f[1:1+n], f[1+n:])
            residual = torch.cat([
                net(a.reshape(-1, a.shape[-1])).reshape(a.shape[0], b, n)
                for net, a in zip(self.networks, groups)], dim=0)
        else:
            residual = self.networks[0](f.reshape(-1, f.shape[-1]))
            residual = residual.reshape(2*n + 1, b, n)
        # Identical Gaussian marginal skip connection in EVERY architecture.
        # This is an explicitly Gaussian-focused prototype, not upstream SΩI.
        base = (sigma / (1 + sigma.square()) * y)[None]
        scale = (sigma / (1 + sigma.square()))[None]
        pred = base + scale * residual
        ix = torch.arange(n, device=x.device)
        j = pred[0]
        m = pred[1+ix, :, ix].T
        c = pred[1+n+ix, :, ix].T
        return j, m, c


@torch.no_grad()
def evaluate(model, x_np, cov, cfg, grid_size=None, seed=8888):
    """SE is conditional on the frozen model; rows are independent clusters."""
    model.eval()
    device = next(model.parameters()).device
    grid_size = grid_size or cfg["grid_size"]
    grid = np.linspace(np.log(cfg["sigma_min"]), np.log(cfg["sigma_max"]), grid_size)
    weights = trapezoid_weights(grid)
    rng = np.random.default_rng(seed)
    n = cov.shape[0]
    precision = np.linalg.inv(cov)
    cond_var = 1 / np.diag(precision)
    cond_mean = x_np - (x_np @ precision) * cond_var
    sums = {}
    curves = []
    for u, weight in zip(grid, weights):
        sigma = float(np.exp(u))
        z = rng.normal(size=x_np.shape)
        y = x_np + sigma * z
        oj = sigma * y @ np.linalg.inv(cov + sigma**2 * np.eye(n))
        om = sigma * y / (np.diag(cov) + sigma**2)
        oc = sigma * (y - cond_mean) / (cond_var + sigma**2)
        predictions = [[], [], []]
        for start in range(0, len(x_np), cfg["eval_batch"]):
            end = start + cfg["eval_batch"]
            xt = torch.as_tensor(x_np[start:end], dtype=torch.float32, device=device)
            zt = torch.as_tensor(z[start:end], dtype=torch.float32, device=device)
            st = torch.full((len(xt), 1), sigma, device=device)
            for out, buf in zip(model(xt, zt, st), predictions):
                buf.append(out.cpu().numpy().astype(np.float64))
        j, m, c = [np.concatenate(a) for a in predictions]
        sq = lambda a: np.square(a).sum(axis=1)
        dtc = sq(j - c)
        tc = sq(j - m)
        rm, rj, rc = sq(z - m), sq(z - j), sq(z - c)
        oracle_direct = sq(oj - om) - sq(oj - oc)
        oracle_risk = sq(z - om) + sq(z - oc) - 2 * sq(z - oj)
        ej, em, ec = sq(j - oj), sq(m - om), sq(c - oc)
        q = dict(direct=tc-dtc, risk=rm+rc-2*rj, tc=tc, dtc=dtc,
                 risk_tc=rm-rj, risk_dtc=rj-rc,
                 oracle_direct=oracle_direct, oracle_risk=oracle_risk,
                 error_joint=ej, error_marginal=em, error_conditional=ec,
                 predicted_risk_bias=em+ec-2*ej,
                 identity_residual=rm+rc-2*rj-oracle_risk-(em+ec-2*ej),
                 consistency_gap=sq(m-c)-tc-dtc,
                 direct_minus_oracle=tc-dtc-oracle_direct,
                 risk_minus_oracle=rm+rc-2*rj-oracle_risk)
        for name, values in q.items():
            if name not in sums:
                sums[name] = np.zeros(len(x_np), dtype=np.float64)
            sums[name] += weight * values
        curves.append({"sigma": sigma, **{k: float(v.mean()) for k, v in q.items()}})
    summary = {}
    for k, v in sums.items():
        summary[k] = float(v.mean())
        summary[k + "_se"] = float(v.std(ddof=1) / np.sqrt(len(v)))
    summary.update({"truth_"+k: v for k, v in gaussian_truth(cov).items()})
    summary.update({"oracle_grid_"+k: v for k, v in oracle_integral(
        cov, cfg["sigma_min"], cfg["sigma_max"], grid_size).items()})
    return summary, pd.DataFrame(curves), sums


In [ ]:
def atomic_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


def atomic_csv(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def training_plot(history, truth, path=None):
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h.step, h.loss, color="#365bb6")
    ax[0].set(xlabel="Step", ylabel="Noise prediction MSE", title="Training")
    if "val_direct" in h:
        valid = h.dropna(subset=["val_direct"])
        ax[1].plot(valid.step, valid.val_direct, "o-", label="Squared differences")
        ax[1].plot(valid.step, valid.val_risk, "o-", label="Risk differences")
    ax[1].axhline(truth, color="black", linestyle="--", label="True Omega")
    ax[1].set(xlabel="Step", ylabel="Omega (nats)", title="Validation only")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=140)
    return fig


def run_one(cfg, case, seed, method, n_train, root, device, live=True):
    cov = covariance(case)
    n = len(cov)
    width = matched_width(cfg["width"], n) if method == "split_budget" else cfg["width"]
    split = method != "shared"
    numerical_cfg = {k: v for k, v in cfg.items()
                     if k not in ("cases", "train_sizes", "seeds", "methods")}
    job = dict(version=VERSION, case=case, seed=seed, method=method,
               n_train=n_train, width=width, config=numerical_cfg,
               torch_version=str(torch.__version__), device=str(device))
    run_id = hashlib.sha256(json.dumps(job, sort_keys=True).encode()).hexdigest()[:12]
    folder = Path(root) / f"{case}_n{n_train}_s{seed}_{method}_{run_id}"
    folder.mkdir(parents=True, exist_ok=True)
    result_path = folder / "result.json"
    if result_path.exists():
        print("Already finished:", folder.name)
        return json.loads(result_path.read_text())
    atomic_json(folder / "config.json", job)
    torch.manual_seed(1000 + seed)
    model = Denoiser(n, width, split, cfg["sigma_min"], cfg["sigma_max"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    data_rng = np.random.default_rng(10000 + seed)
    x_train = data_rng.multivariate_normal(np.zeros(n), cov, n_train).astype(np.float32)
    val_rng = np.random.default_rng(20000 + seed)
    x_val = val_rng.multivariate_normal(np.zeros(n), cov, cfg["n_val"])
    test_rng = np.random.default_rng(30000 + seed)
    x_test = test_rng.multivariate_normal(np.zeros(n), cov, cfg["n_test"])
    # Separate RNG ensures identical minibatches/time/noise for all architectures.
    rng = np.random.default_rng(40000 + seed)
    checkpoint = folder / "checkpoint.pt"
    start, history, elapsed_train = 0, [], 0.
    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location=device, weights_only=False)
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        rng.bit_generator.state = saved["rng_state"]
        start, history, elapsed_train = saved["step"], saved["history"], saved["train_seconds"]
        print("Resume from step", start)
    title = f"{case} / seed {seed} / {method}"
    progress = tqdm(range(start, cfg["steps"]), desc=title, total=cfg["steps"], initial=start)
    loss_sum, loss_count = 0., 0
    sync_device(device)
    train_clock = time.perf_counter()
    for step0 in progress:
        model.train()
        ids = rng.integers(n_train, size=cfg["batch_size"])
        noise = rng.normal(size=(cfg["batch_size"], n)).astype(np.float32)
        u = rng.uniform(np.log(cfg["sigma_min"]), np.log(cfg["sigma_max"]),
                        size=(cfg["batch_size"], 1))
        x = torch.as_tensor(x_train[ids], device=device)
        z = torch.as_tensor(noise, device=device)
        sigma = torch.as_tensor(np.exp(u).astype(np.float32), device=device)
        j, m, c = model(x, z, sigma)
        loss = ((j-z).square().mean() + (m-z).square().mean() + (c-z).square().mean()) / 3
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss in {folder}; failed runs are not discarded")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
        optimizer.step()
        loss_sum += float(loss.detach().cpu())
        loss_count += 1
        step = step0 + 1
        if step % cfg["log_every"] == 0 or step == cfg["steps"]:
            sync_device(device)
            elapsed_train += time.perf_counter() - train_clock
            row = dict(step=step, loss=loss_sum/loss_count, train_seconds=elapsed_train)
            if step % cfg["validate_every"] == 0 or step == cfg["steps"]:
                val, _, _ = evaluate(model, x_val, cov, cfg, grid_size=16, seed=50000+seed)
                row.update(val_direct=val["direct"], val_risk=val["risk"])
            history.append(row)
            loss_sum, loss_count = 0., 0
            tmp = checkpoint.with_suffix(".tmp")
            torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                            rng_state=rng.bit_generator.state, step=step, history=history,
                            train_seconds=elapsed_train), tmp)
            os.replace(tmp, checkpoint)
            atomic_csv(folder / "training.csv", pd.DataFrame(history))
            if hasattr(progress, "set_postfix"):
                progress.set_postfix(loss=round(row["loss"], 4))
            if "val_direct" in row:
                fig = training_plot(history, gaussian_truth(cov)["omega"], folder / "training.png")
                if live:
                    try:
                        from IPython.display import display
                        if "handle" not in locals():
                            handle = display(fig, display_id=True)
                        elif handle is not None:
                            handle.update(fig)
                    except ImportError:
                        pass
                plt.close(fig)
            sync_device(device)
            train_clock = time.perf_counter()
    summary, curves, observations = evaluate(model, x_test, cov, cfg, seed=60000+seed)
    summary.update(case=case, seed=seed, method=method, n_train=n_train,
                   parameters=sum(p.numel() for p in model.parameters()), width=width,
                   steps=cfg["steps"], train_seconds=elapsed_train, folder=str(folder))
    # Classical baseline uses training data only; no knowledge of the true covariance.
    sample_cov = np.cov(x_train.astype(np.float64), rowvar=False, ddof=1)
    summary["gaussian_plugin"] = gaussian_truth(sample_cov)["omega"]
    summary["direct_abs_error"] = abs(summary["direct"] - summary["truth_omega"])
    summary["risk_abs_error"] = abs(summary["risk"] - summary["truth_omega"])
    atomic_csv(folder / "integrands.csv", curves)
    np.savez_compressed(folder / "observations.npz", **observations)
    # result.json is written LAST: existence means the whole job completed.
    atomic_json(result_path, summary)
    return summary


def result_plot(results, root=None):
    df = pd.DataFrame(results)
    fig, axes = plt.subplots(1, 2, figsize=(12, max(4, len(df)*.22)))
    labels = [f"{r.case}, n={r.n_train}, s={r.seed}, {r.method}" for r in df.itertuples()]
    y = np.arange(len(df))
    for offset, estimator, color in [(-.13, "direct", "#365bb6"), (.13, "risk", "#d35c38")]:
        axes[0].errorbar(df[estimator], y+offset, xerr=1.96*df[estimator+"_se"],
                         fmt="o", markersize=4, label=estimator, color=color)
    axes[0].scatter(df.truth_omega, y, marker="x", color="black", label="Truth")
    axes[0].scatter(df.gaussian_plugin, y, marker="|", color="#399563", label="Gaussian plug-in")
    axes[0].set_yticks(y, labels, fontsize=7)
    axes[0].set(xlabel="Omega (nats)", title="Bars: evaluation uncertainty ONLY")
    axes[0].legend(fontsize=8)
    axes[1].scatter(df.parameters, df.direct_abs_error, label="direct")
    axes[1].scatter(df.parameters, df.risk_abs_error, label="risk")
    axes[1].set(xlabel="Parameter count", ylabel="Absolute error (nats)",
                title="Accuracy / parameter budget")
    axes[1].legend()
    fig.tight_layout()
    if root:
        fig.savefig(Path(root)/"comparison.png", dpi=160)
    return fig


def run_suite(cfg, root, device, live=True):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    atomic_json(root / "environment.json", dict(version=VERSION, python=platform.python_version(),
                torch=str(torch.__version__), numpy=np.__version__, device=str(device)))
    results = []
    for case in cfg["cases"]:
        for n_train in cfg["train_sizes"]:
            for seed in cfg["seeds"]:
                for method in cfg["methods"]:
                    try:
                        row = run_one(cfg, case, seed, method, n_train, root, device, live=live)
                    except Exception as error:
                        failures = root / "failures.jsonl"
                        with failures.open("a", encoding="utf-8") as f:
                            f.write(json.dumps(dict(case=case, n_train=n_train, seed=seed,
                                method=method, error=repr(error)), ensure_ascii=False)+"\n")
                        raise
                    results.append(row)
                    atomic_csv(root / "results.csv", pd.DataFrame(results))
                    frame = pd.DataFrame(results)
                    aggregate = frame.groupby(["case", "n_train", "method"], as_index=False).agg(
                        seeds=("seed", "count"), direct_mean=("direct", "mean"),
                        direct_sd=("direct", "std"), risk_mean=("risk", "mean"),
                        risk_sd=("risk", "std"), direct_mae=("direct_abs_error", "mean"),
                        risk_mae=("risk_abs_error", "mean"), truth=("truth_omega", "first"))
                    atomic_csv(root / "summary.csv", aggregate)
                    fig = result_plot(results, root)
                    if live:
                        try:
                            from IPython.display import display
                            display(fig)
                        except ImportError:
                            pass
                    plt.close(fig)
                    print(f"Saved {len(results)} runs; {method}: direct={row['direct']:.4f}, "
                          f"risk={row['risk']:.4f}, truth={row['truth_omega']:.4f}")
    return pd.DataFrame(results)


## Контроль без обучения

`oracle_grid` вычисляет ожидаемые подынтегральные величины аналитически, затем применяет
тот же диапазон и сетку интегрирования. Разность с истиной показывает численное смещение,
которое останется даже при идеальной сети. Это отдельный контроль, а не обучаемый baseline.

Варианты данных:

| Имя | Смысл |
|---|---|
| `pair_null` | Три независимые пары с корреляцией 0.9 внутри каждой; $TC=DTC>0$, $\Omega=0$. |
| `independent` | Шесть независимых нормальных переменных; все три величины равны нулю. |
| `redundant` | Две независимые избыточные тройки; $\Omega>0$. |
| `synergistic` | Две независимые тройки $(A,B,(A+B+\varepsilon)/s)$; $\Omega<0$. |
| `balanced` | Избыточная и синергетическая тройки с взаимной компенсацией; $\Omega=0$. |
| `mixed_positive` | Та же конструкция, чуть больше избыточности; $\Omega\approx0.0903$ нат. |
| `mixed_negative` | Чуть меньше избыточности; $\Omega\approx-0.0731$ нат. |

Нулевое значение О-информации само по себе не означает независимость. В `balanced` оно
сосуществует с ненулевыми вкладами обоих типов. Эту интерпретацию нельзя смешивать с ошибкой оценивания.


In [ ]:
CONTROL_CASES = ["pair_null", "independent", "redundant", "synergistic",
                 "balanced", "mixed_positive", "mixed_negative"]
control_rows = []
for case in CONTROL_CASES:
    cov = covariance(case)
    exact = gaussian_truth(cov)
    approximate = oracle_integral(cov, .005, 100., 48)
    control_rows.append(dict(case=case, **exact,
                            oracle_grid=approximate["omega"],
                            numerical_shift=approximate["omega"]-exact["omega"]))
controls = pd.DataFrame(control_rows)
display(controls.round(6))
controls.to_csv(OUTPUT_ROOT/"oracle_controls.csv", index=False)


## Панель запуска

Начните с `pilot`: два распределения, один seed, три архитектуры, 2500 шагов.
Короткий запуск `smoke` проверяет только работу кода. `extended` увеличивает число шагов
и повторений; время на разных устройствах зависит от аппаратуры.
Прежде чем делать вывод, посмотрите, стабилизировались ли валидационные кривые, и повторите
эксперимент как минимум с 5 seed. Если они не стабилизируются, увеличьте бюджет.

Прерывание — кнопка остановки/Interrupt в Jupyter или Colab. Повторный запуск с теми же
настройками загрузит checkpoint; завершённые задания пропускаются. Добавление новых seed
или распределений не требует пересчитывать старые задания. Изменение численных настроек
создаёт новую серию каталогов, сохраняя старые результаты.

В маленькой сети GPU не всегда быстрее CPU. Режим CPU использует не более 4 потоков.
Реплики здесь меняют и обучающую выборку, и инициализацию. Их разброс — суммарная вариабельность,
а не отдельная оценка только ошибки конечной выборки.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, FileLink
import shutil

device_options = ["cpu"]
if torch.cuda.is_available():
    device_options.insert(0, "cuda")
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_options.insert(0, "mps")

style = {"description_width": "150px"}
profile_ui = widgets.Dropdown(options=["smoke", "pilot", "extended"], value="pilot",
                              description="Режим", style=style)
case_ui = widgets.SelectMultiple(options=CONTROL_CASES, value=("pair_null", "mixed_positive"),
                                 description="Распределения", rows=7, style=style)
steps_ui = widgets.BoundedIntText(value=2500, min=50, max=100000, description="Шаги", style=style)
seeds_ui = widgets.BoundedIntText(value=1, min=1, max=30, description="Количество seed", style=style)
size_ui = widgets.Text(value="5000", description="Размеры train", style=style,
                       placeholder="500, 2000, 10000")
device_ui = widgets.Dropdown(options=device_options, value=device_options[0],
                             description="Устройство", style=style)
run_button = widgets.Button(description="Запустить / продолжить", button_style="primary",
                            layout=widgets.Layout(width="230px"))
download_button = widgets.Button(description="Скачать ZIP", layout=widgets.Layout(width="180px"))
ui_output = widgets.Output()

def set_profile(change):
    steps_ui.value, seeds_ui.value = {"smoke": (300, 1), "pilot": (2500, 1),
                                      "extended": (8000, 5)}[change["new"]]

profile_ui.observe(set_profile, names="value")

def selected_config():
    sizes = [int(s.strip()) for s in size_ui.value.split(",") if s.strip()]
    if not sizes or min(sizes) < 50 or max(sizes) > 1000000:
        raise ValueError("Укажите размеры train от 50 до 1000000")
    if not case_ui.value:
        raise ValueError("Выберите хотя бы одно распределение")
    return dict(width=64, sigma_min=.005, sigma_max=100., grid_size=48,
                eval_batch=256, steps=steps_ui.value, lr=.002, batch_size=64,
                log_every=250, validate_every=500, n_val=256, n_test=2048,
                cases=list(case_ui.value), train_sizes=sizes,
                seeds=list(range(seeds_ui.value)),
                methods=["shared", "split_budget", "split_full"])

def start_experiment(_=None):
    global RESULTS, LAST_CONFIG
    run_button.disabled = True
    try:
        with ui_output:
            ui_output.clear_output(wait=True)
            cfg = selected_config()
            LAST_CONFIG = cfg
            device = torch.device(device_ui.value)
            if device.type == "cpu":
                torch.set_num_threads(min(4, os.cpu_count() or 1))
            total = len(cfg["cases"])*len(cfg["train_sizes"])*len(cfg["seeds"])*len(cfg["methods"])
            print(f"Заданий: {total}. Устройство: {device}. Результаты: {OUTPUT_ROOT}")
            print("Интервалы на графике учитывают только оценивание при фиксированной сети.")
            RESULTS = run_suite(cfg, OUTPUT_ROOT, device, live=True)
            display(RESULTS[["case", "seed", "method", "parameters", "direct", "risk",
                             "truth_omega", "direct_abs_error", "risk_abs_error", "train_seconds"]])
            print("Готово. Файлы сохранены после каждого задания.")
    finally:
        run_button.disabled = False

def download_archive(_=None):
    with ui_output:
        zip_base = Path("/content/soi_results") if IN_COLAB else Path.cwd()/"soi_results"
        archive = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_ROOT)
        if IN_COLAB:
            from google.colab import files
            files.download(archive)
        else:
            display(FileLink(archive))

run_button.on_click(start_experiment)
download_button.on_click(download_archive)
display(widgets.VBox([profile_ui, case_ui, steps_ui, seeds_ui, size_ui, device_ui,
                      widgets.HBox([run_button, download_button]), ui_output]))
print("Если виджеты не отображаются: в новой ячейке выполните start_experiment().")


## Что сохраняется и как читать результаты

- `results.csv` — каждый завершённый запуск: оценка, истина, ошибка, число параметров, время.
- `summary.csv` — средние и стандартные отклонения между seed; при одном seed SD отсутствует.
- `comparison.png` — оценки и абсолютные ошибки.
- В каталоге каждого задания: `config.json`, `checkpoint.pt`, `training.csv`, `training.png`,
  `integrands.csv`, `observations.npz`, `result.json`.
- `failures.jsonl` — ошибки запусков, если они возникнут. Они не отбрасываются молча.

**Четыре сравнения:**

1. `shared` против `split_budget`: отделение задач при близком общем числе параметров.
2. `shared` против `split_full`: что даёт отделение вместе с увеличением общей ёмкости.
3. `direct` против `risk` внутри одного запуска: меняется только способ вычисления оценки.
4. Оба результата против `oracle_grid_omega`: насколько мешает приближение УМО,
   если отдельно учесть квадратуру и отсечение диапазона.

`error_joint/marginal/conditional` — интегрированные ошибки относительно истинных гауссовских УМО.
`predicted_risk_bias` — $E_M+E_C-2E_J$.
`identity_residual` — проверка выведенного разложения по парным тестовым наблюдениям;
её ожидание равно нулю, но конечная оценка шумит. Сравнивайте с `identity_residual_se`.
`consistency_gap` сравнивает три квадратичные формулы: $S-TC-DTC$; с точными УМО
его ожидание равно нулю, с нейросетями это дополнительный диагностический сигнал.

`gaussian_plugin` — оценка по выборочной ковариации обучающих данных. Это сильный
специализированный baseline для этого семейства, но его успех ничего не доказывает
о качестве методов на негауссовских распределениях.

Столбцы `*_se` и полосы на графиках отражают **только тестовые данные и добавляемый шум при
фиксированной обученной модели**. Они не покрывают ошибку обучения, смещение модели и
отсечение интеграла. Не называйте их доверительными интервалами для истинной $\Omega$.

При $\Omega\approx0$ используйте абсолютную ошибку; относительная ошибка плохо определена.
Не присваивайте знак статистически неразличимым от нуля значениям. Для заявления о преимуществе
нужны повторения и сопоставимый бюджет; стоит отдельно сравнить одинаковое время обучения.
Конкретный пример с плохим результатом не доказывает глобальную несостоятельность метода.


## Следующие небольшие эксперименты

1. **Размер выборки:** на панели задайте `500, 2000, 10000`; используйте одинаковый бюджет шагов.
   Затем отдельно проверьте, не ограничивает ли результат недообучение при большом train.
2. **Окрестность смены знака:** выберите `balanced`, `mixed_positive`, `mixed_negative`.
   Оцените абсолютную ошибку и устойчивость классификации знака между seed.
3. **Причина выигрыша:** если `split_full` выигрывает, но `split_budget` — нет,
   результат совместим с объяснением через общую ёмкость. Сам по себе он не устанавливает
   вред от совместного обучения. Если `split_budget` выигрывает устойчиво, гипотеза о
   конфликте задач получает более сильную поддержку, но остаются различия оптимизации.
4. **Проверка исходной статьи:** переносите удачный тест в официальный репозиторий SΩI,
   сохраняя его архитектуру, преобразования, обучение, интегрирование и настройки baseline.
   Этот ноутбук не обучает MINE/InfoNCE/Line-MINDE и не доказывает проигрыш оригинального SΩI.
5. **Негауссовский этап:** покомпонентные обратимые преобразования сохраняют истинную
   О-информацию. Для них нужно заменить гауссовский oracle и убрать Gaussian-specific skip.
   Сама проверка нелинейных преобразований уже есть в приложении E исходной статьи;
   возможное продолжение — систематическая проверка устойчивости и ошибки знака.

## Источники и границы новизны

- [Bounoua et al., SΩI, 2024](https://arxiv.org/abs/2402.05667),
  [полный текст](https://arxiv.org/html/2402.05667v3),
  [официальный код](https://github.com/MustaphaBounoua/soi).
- [Yu et al., MMG](https://arxiv.org/abs/2509.20609): оценивание взаимной информации
  через разность MMSE. Общая идея такой разности уже известна; здесь рассматривается
  её применение к О-информации и совместное обучение разных УМО.
- [Gehlen et al., Bias in O-Information Estimation, 2024](https://pmc.ncbi.nlm.nih.gov/articles/PMC11507536/):
  смещение оценок дискретной О-информации. Его выводы нельзя автоматически переносить на SΩI.

Предложенное продолжение — исследование ошибки, разделения задач и альтернативной формулы
в контролируемом протоколе. По этому ноутбуку нельзя заявлять мировую новизну или
универсальное превосходство. Полезный отрицательный результат также может войти в анализ статьи.

Техническая проверка перед передачей: выполнены все ячейки настройки и интерфейса,
шесть CPU-запусков по 2500 шагов и проверка контрольных гауссовских систем.
Искусственное прерывание после checkpoint и последующее продолжение дали те же оценки,
что непрерывный CPU-запуск. CUDA, MPS и подключение Google Drive здесь не проверялись.
